In [3]:
import requests
import json

In [8]:
import requests
import json
import time

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
    "https://overpass.osm.ch/api/interpreter",
]

BBOX = "18.900,72.780,19.250,72.990"
AMENITIES = ["pharmacy", "hospital", "atm", "parking", "restaurant", "fuel", "cafe"]

headers = {
    "User-Agent": "here-nlp-search/1.0 (educational portfolio project)"
}

all_places = []
seen = set()

def fetch_one_amenity(amenity):
    query = f"""
    [out:json][timeout:20];
    node["amenity"="{amenity}"]({BBOX});
    out;
    """
    last_error = None

    for url in OVERPASS_URLS:
        try:
            print(f"Trying {amenity} via {url}")
            response = requests.post(url, data=query, headers=headers, timeout=40)
            print("Status:", response.status_code)

            response.raise_for_status()
            data = response.json()

            count = 0
            for element in data.get("elements", []):
                tags = element.get("tags", {})
                name = tags.get("name", "").strip()
                if not name:
                    continue

                lat = element.get("lat")
                lon = element.get("lon")
                key = (name, lat, lon)
                if key in seen:
                    continue
                seen.add(key)

                all_places.append({
                    "name": name,
                    "category": amenity,
                    "lat": lat,
                    "lon": lon,
                    "desc": f"{name} is a {amenity} in Mumbai."
                })
                count += 1

            print(f"Added {count} named {amenity} places")
            return

        except Exception as e:
            last_error = e
            print(f"Failed on {url}: {e}")
            time.sleep(3)

    raise RuntimeError(f"All endpoints failed for amenity={amenity}: {last_error}")

for amenity in AMENITIES:
    fetch_one_amenity(amenity)
    time.sleep(2)

print(f"Total places collected: {len(all_places)}")

with open("places.json", "w", encoding="utf-8") as f:
    json.dump(all_places, f, ensure_ascii=False, indent=2)

print("Saved to places.json")

Trying pharmacy via https://overpass-api.de/api/interpreter
Status: 504
Failed on https://overpass-api.de/api/interpreter: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
Trying pharmacy via https://overpass.kumi.systems/api/interpreter
Status: 200
Added 144 named pharmacy places
Trying hospital via https://overpass-api.de/api/interpreter
Status: 504
Failed on https://overpass-api.de/api/interpreter: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
Trying hospital via https://overpass.kumi.systems/api/interpreter
Failed on https://overpass.kumi.systems/api/interpreter: HTTPSConnectionPool(host='overpass.kumi.systems', port=443): Read timed out. (read timeout=40)
Trying hospital via https://overpass.openstreetmap.ru/api/interpreter
Failed on https://overpass.openstreetmap.ru/api/interpreter: HTTPSConnectionPool(host='overpass.openstreetmap.ru', port=443): Max retries exceeded with url: /api/interpreter (Caused by Conne

In [22]:
import requests
import json
import time

OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.openstreetmap.ru/api/interpreter",
    "https://overpass.osm.ch/api/interpreter",
]

BBOX = "18.900,72.780,19.250,72.990"
AMENITIES = ["fuel", "hospital"]  # only missing ones

headers = {"User-Agent": "here-nlp-search/1.0 (educational portfolio project)"}

all_places = []
seen = set()

def fetch_one_amenity(amenity):
    query = f"""
        [out:json][timeout:60];

        (
        node["amenity"="{amenity}"]({BBOX});
        way["amenity"="{amenity}"]({BBOX});
        relation["amenity"="{amenity}"]({BBOX});
        );

        out center tags;
        """
    last_error = None
    for url in OVERPASS_URLS:
        try:
            print(f"Trying {amenity} via {url}")
            response = requests.post(url, data=query, headers=headers, timeout=40)
            response.raise_for_status()
            data = response.json()
            count = 0
            for element in data.get("elements", []):
                tags = element.get("tags", {})
                name = tags.get("name", "").strip()
                if not name:
                    continue
                lat = element.get("lat")
                lon = element.get("lon")
                key = (name, lat, lon)
                if key in seen:
                    continue
                seen.add(key)
                all_places.append({
                    "name": name,
                    "category": amenity,
                    "lat": lat,
                    "lon": lon,
                    "desc": f"{name} is a {amenity} in Mumbai."
                })
                count += 1
            print(f"Added {count} named {amenity} places")
            return
        except Exception as e:
            last_error = e
            print(f"Failed on {url}: {e}")
            time.sleep(3)
    raise RuntimeError(f"All endpoints failed for {amenity}: {last_error}")

for amenity in AMENITIES:
    fetch_one_amenity(amenity)
    time.sleep(2)

print(f"Newly fetched: {len(all_places)}")

Trying fuel via https://overpass-api.de/api/interpreter
Added 75 named fuel places
Trying hospital via https://overpass-api.de/api/interpreter
Failed on https://overpass-api.de/api/interpreter: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter
Trying hospital via https://overpass.kumi.systems/api/interpreter
Failed on https://overpass.kumi.systems/api/interpreter: HTTPSConnectionPool(host='overpass.kumi.systems', port=443): Read timed out. (read timeout=40)
Trying hospital via https://overpass.openstreetmap.ru/api/interpreter
Failed on https://overpass.openstreetmap.ru/api/interpreter: HTTPSConnectionPool(host='overpass.openstreetmap.ru', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<HTTPSConnection(host='overpass.openstreetmap.ru', port=443) at 0x113ab71f0>, 'Connection to overpass.openstreetmap.ru timed out. (connect timeout=40)'))
Trying hospital via https://overpass.osm.ch/api/interpreter
Added 0 named ho

In [23]:
with open("places.json") as f:
    existing = json.load(f)

combined = existing + all_places

seen_keys = set()
final = []
for p in combined:
    key = (p["name"], p["lat"], p["lon"])
    if key not in seen_keys:
        seen_keys.add(key)
        final.append(p)

with open("places.json", "w", encoding="utf-8") as f:
    json.dump(final, f, ensure_ascii=False, indent=2)

print(f"Total places now: {len(final)}")

Total places now: 1831


In [21]:
import json
import random
from collections import Counter

MAX_HOSPITALS = 700

with open("places.json", "r", encoding="utf-8") as f:
    places = json.load(f)

# Split hospitals and everything else
hospitals = [p for p in places if p["category"] == "hospital"]
others = [p for p in places if p["category"] != "hospital"]

print("Before:")
print(Counter(p["category"] for p in places))

# Downsample hospitals
if len(hospitals) > MAX_HOSPITALS:
    hospitals = random.sample(hospitals, MAX_HOSPITALS)

# Merge back
final_places = others + hospitals

# Save
with open("places.json", "w", encoding="utf-8") as f:
    json.dump(final_places, f, ensure_ascii=False, indent=2)

print("\nAfter:")
print(Counter(p["category"] for p in final_places))
print(f"\nTotal places: {len(final_places)}")

Before:
Counter({'restaurant': 727, 'hospital': 400, 'cafe': 351, 'pharmacy': 144, 'atm': 112, 'parking': 22})

After:
Counter({'restaurant': 727, 'hospital': 400, 'cafe': 351, 'pharmacy': 144, 'atm': 112, 'parking': 22})

Total places: 1756
